In [12]:
import rioxarray as rxr 
import numpy as np
import os
import xarray as xr


In [ ]:
# Calculate the EVI difference between pre-fire and post-fire for 2015-2017 and 20215 and 2020

In [5]:
post_fire_evi_2015= rxr.open_rasterio('input-data/outcome-layers/post-fire-evi.tif')
post_fire_evi_2017=rxr.open_rasterio('input-data/outcome-layers/reprojected_EVI_2017.tif')
post_fire_evi_2020=rxr.open_rasterio('input-data/outcome-layers/reprojected_EVI_2020.tif')

In [9]:
layers = {
    "2015": post_fire_evi_2015,
    "2017": post_fire_evi_2017,
    "2020": post_fire_evi_2020,
}

In [ ]:
clean_layers = {}
for year, da in layers.items():
    # keep only theoretical EVI values
    clean = da.where((da >= -1) & (da <= 1))
    clean_layers[year] = clean

    before = np.isfinite(da.values).sum()
    after = np.isfinite(clean.values).sum()
    print(f"{year}: removed {before - after} outlier pixels")

# optional: replace original variablesi 
post_fire_evi_2015 = clean_layers["2015"]
post_fire_evi_2017 = clean_layers["2017"]
post_fire_evi_2020 = clean_layers["2020"]

2015: removed 1 outlier pixels
2017: removed 172 outlier pixels
2020: removed 1 outlier pixels


In [ ]:
# Align rasters and keep only overlapping valid pixels
evi2015 = clean_layers["2015"]
evi2017 = clean_layers["2017"]
evi2020 = clean_layers["2020"]

evi2015_s, evi2017_s = xr.align(evi2015, evi2017, join="inner")
evi2015_l, evi2020_l = xr.align(evi2015, evi2020, join="inner")

mask_short = np.isfinite(evi2015_s) & np.isfinite(evi2017_s)
mask_long = np.isfinite(evi2015_l) & np.isfinite(evi2020_l)

dif_evi_short = (evi2017_s - evi2015_s).where(mask_short).astype("float32")  # 2017 - 2015
dif_evi_long = (evi2020_l - evi2015_l).where(mask_long).astype("float32")    # 2020 - 2015

# Create: output-data/output_EVI_dif
out_dir = "output-data/output_EVI_dif"
os.makedirs(out_dir, exist_ok=True)

# Save outputs
dif_evi_short.rio.to_raster(f"{out_dir}/dif_evi_short_2017_minus_2015.tif")
dif_evi_long.rio.to_raster(f"{out_dir}/dif_evi_long_2020_minus_2015.tif")


Saved files in: output-data/output_EVI_dif
